In [30]:
import pandas as pd
import re
import folium
from folium import plugins
from collections import defaultdict
from fuzzywuzzy import fuzz, process
import numpy as np


In [31]:

# Загрузка данных
print("📂 Загрузка данных...")
df_products = pd.read_csv('vkusvill_itog_2026-05-19.csv', sep=',')
df_cities = pd.read_csv('city.csv', sep=',')


📂 Загрузка данных...


In [32]:
df_products

,Unnamed: 0,category,subcategory,price,price_unit,weight,rating,description,calories,proteins,...,carbs,shelf_life,brand,storage_conditions,manufacturer,country,composition,labels,url,parsed_at
0,"Салфетки бумажные 2сл 85 листов, в салфетнице",Товары для дома и кухни,"Бумажные полотенца, салфетки, зубочистки",123.0,шт,шт,5,Двухслойные бумажные салфетки с рисунком. Не т...,NaN,NaN,...,NaN,ТИШЬЮПРОМ ООО: Не ограничен;,ВкусВилл,ТИШЬЮПРОМ ООО: Хранить при температуре от 2 до...,"ООО ""ТИШЬЮПРОМ"":Россия,141305, Московская обл,...",ТИШЬЮПРОМ ООО: РОССИЯ;,"ООО ""ТИШЬЮПРОМ"":100 % целлюлоза первичное воло...",Разный дизайн,https://vkusvill.ru/goods/salfetki-bumazhnye-2...,2026-05-19 11:27:38
1,"Мармелад жевательный ""Лесные друзья"", 100 г",Сладости и десерты,"Зефир, пастила, безе и мармелад",104.0,шт,100 г,4.9,"Ассорти мармеладных мишек с яблочным, вишнёвым...",320.9,5.0,...,75.0,КФ МАРМИ ООО: 12 Месяцев,ВкусВилл,КФ МАРМИ ООО: Хранить при температуре от 15 до...,"КФ МАРМИ ООО:Россия, 142033, Московская област...",КФ МАРМИ ООО: РОССИЯ;,"КФ МАРМИ ООО:глюкозный сироп, сахар, желатин п...",NaN,https://vkusvill.ru/goods/marmelad-zhevatelnyy...,2026-05-19 11:27:38
2,"Голубцы, зам.",Замороженные продукты,Семейный формат,395.0,шт,600 г,4.9,Голубцы по домашнему рецепту. Тонкие капустные...,123.5,7.0,...,7.0,Кузнецов Иван Владимирович ИП (526014180520): ...,ВкусВилл,Кузнецов Иван Владимирович ИП (526014180520): ...,NaN,Кузнецов Иван Владимирович ИП (526014180520): ...,ИП Кузнецов Иван Владимирович:капуста белокоча...,"Большая порция, Заморозка",https://vkusvill.ru/goods/golubtsy-1147/,2026-05-19 11:27:38
3,"Лимонад ""Домашний"" с соком яблока-груши, 330 мл",Напитки,"Лимонады, газировка",150.0,шт,330 мл,4.9,Освежающий напиток с грушевым и яблочным соком...,36.0,NaN,...,9.0,ДОМАШНИЙ+ ООО: 90 Суток,ВкусВилл,ДОМАШНИЙ+ ООО: Хранить при температуре от 2 до...,"ООО ""ДОМАШНИЙ+"":Россия, 602264, Владимирская о...",ДОМАШНИЙ+ ООО: РОССИЯ;,"ООО ""ДОМАШНИЙ+"":вода питьевая, инвертный сироп...",Светофор,https://vkusvill.ru/goods/limonad-domashniy-s-...,2026-05-19 11:27:38
4,Батончик протеиновый с малиной и кешью,Сладости и десерты,Шоколад и батончики,187.0,шт,45 г,4.8,Десертный протеиновый батончик для ценителей п...,355.2,12.7,...,13.1,ДУ ИТ ООО: 180 Суток,ВкусВилл,ДУ ИТ ООО: Хранить при температуре от 2 до 22 ...,"ООО ""ДУ ИТ"":Россия, 115230, город Москва, прое...",ДУ ИТ ООО: РОССИЯ;,"ООО ""ДУ ИТ"":сироп цикория (инулин из корней ци...",Растительная линейка,https://vkusvill.ru/goods/batonchik-proteinovy...,2026-05-19 11:27:38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20170,"Набор листового чая (да хун пао, шу пуэр дворц...",Чай,Черный чай,626.0,шт,45 г,4.8,"Набор чёрного китайского чая, объединивший в с...",NaN,NaN,...,NaN,12 месяцев,Kinby,АПГРЕЙД ТРЕЙДИНГ ООО: Хранить при температуре ...,"ООО ""АПГРЕЙД ТРЕЙДИНГ"":Россия, 117105, город М...",АПГРЕЙД ТРЕЙДИНГ ООО: РОССИЯ;,Чай Да Хун Пао: чай черный китайский листовой ...,NaN,https://vkusvill.ru/goods/nabor-listovogo-chay...,2026-05-19 12:32:13
20171,"Чай зеленый листовой китайский ""Молочный улун""",Индилавка: вкусы мира,Всё для чая и кофе,322.0,шт,50 г,4.9,Зелёный листовой чай «Молочный улун» — частичн...,NaN,NaN,...,NaN,24 месяцев,ВкусВилл,Хунань Сяосян Ти Индастри Ко Лтд: Хранить при ...,"Хунань Сяосян Ти Индастри Ко Лтд:Китай, Номер ...",Хунань Сяосян Ти Индастри Ко Лтд: КИТАЙ;,чай зеленый китайский листовой частично фермен...,NaN,https://vkusvill.ru/goods/chay-zelenyy-listovo...,2026-05-19 12:32:13
20172,"Рис для суши ""Японка""","Крупы, макароны, мука",Крупы,299.0,шт,500 г,4.9,"Рис «Японка» — шлифованный рис экстра-сорта, и...",362.5,6.5,...,83.0,АПГРЕЙД ТРЕЙДИНГ ООО: 12 Месяцев,ВкусВилл,АПГРЕЙД ТРЕЙДИНГ ООО: Хранить при температуре ...,"ООО ""АПГРЕЙД ТРЕЙДИНГ"":Россия, 143200, Московс...",АПГРЕЙД ТРЕЙДИНГ ООО: РОССИЯ;,"ООО ""АПГРЕЙД ТРЕЙДИНГ"":рис шлифованный сорт эк...",Растительная линейка,https://vkusvill.ru/goods/ris-dlya-sushi-yapon...,2026-05-19

In [33]:

# Создаем словарь городов с координатами из city.csv
cities_dict = {}
for idx, row in df_cities.iterrows():
    # Берем название города и региона
    city_name = row['city'] if pd.notna(row['city']) else ''
    region_name = row['region'] if pd.notna(row['region']) else ''
    
    # Координаты
    lat = row['geo_lat'] if pd.notna(row['geo_lat']) else None
    lon = row['geo_lon'] if pd.notna(row['geo_lon']) else None
    
    if lat and lon:
        # Сохраняем по разным ключам для лучшего поиска
        if city_name:
            cities_dict[city_name.lower()] = (lat, lon, f"{city_name}, {region_name}")
        if region_name:
            cities_dict[region_name.lower()] = (lat, lon, f"{city_name}, {region_name}")
        
        # Также сохраняем комбинацию
        if city_name and region_name:
            cities_dict[f"{city_name} {region_name}".lower()] = (lat, lon, f"{city_name}, {region_name}")

print(f"✅ Загружено {len(cities_dict)} записей городов с координатами")


✅ Загружено 2269 записей городов с координатами


In [34]:

def extract_location_from_manufacturer(manufacturer_str):
    """
    Извлекает город и регион из строки manufacturer
    """
    if pd.isna(manufacturer_str):
        return None, None
    
    # Паттерны для поиска локации
    patterns = [
        # Город после "г."
        r'(?:г\.|город)\s+([А-ЯЁ][а-яё]+(?:ск|ль|ов|ев|ин|град)?)',
        # Область/край
        r'([А-ЯЁ][а-яё]+(?:ская|ский|овская|евская|инская))(?:\s+область|\s+край)',
        # Республика
        r'(?:Республика)\s+([А-ЯЁ][а-яё]+)',
        # Прямое название города в конце адреса
        r',\s*([А-ЯЁ][а-яё]+(?:ск|ль|ов|ев|ин|град)?)(?:\s*[,)]|$)',
        # Название после области
        r'область[^,]*,\s*([А-ЯЁ][а-яё]+(?:ск|ль|ов|ев|ин|град)?)',
    ]
    
    for pattern in patterns:
        match = re.search(pattern, manufacturer_str)
        if match:
            location = match.group(1).strip()
            return location, location
    
    # Проверка на страны (если нет российского адреса)
    countries = ['РОССИЯ', 'Россия', 'БЕЛАРУСЬ', 'ГРУЗИЯ', 'АЗЕРБАЙДЖАН', 'ТУРЦИЯ', 'КИТАЙ', 'ГРЕЦИЯ']
    for country in countries:
        if country in manufacturer_str.upper():
            return country, country
    
    return None, None


In [35]:

def find_city_coordinates(city_name, region_name=None):
    """
    Находит координаты города в city.csv
    Использует fuzzy matching для неточных совпадений
    """
    if not city_name:
        return None, None, None
    
    city_name_lower = city_name.lower()
    
    # Прямое совпадение
    if city_name_lower in cities_dict:
        lat, lon, full_name = cities_dict[city_name_lower]
        return lat, lon, full_name
    
    # Поиск частичного совпадения
    best_match = None
    best_score = 0
    
    for key, (lat, lon, full_name) in cities_dict.items():
        # Используем fuzzy matching
        score = fuzz.ratio(city_name_lower, key)
        if score > best_score and score > 70:  # Порог 70%
            best_score = score
            best_match = (lat, lon, full_name)
    
    if best_match:
        return best_match
    
    return None, None, None


In [ ]:

# Извлечение локаций из продуктов
print("\n🔍 Извлечение локаций из производителей...")
df_products['city_extracted'], df_products['region_extracted'] = zip(*df_products['manufacturer'].apply(extract_location_from_manufacturer))

# Группировка продуктов по городам
city_products = defaultdict(lambda: {'products': [], 'ratings': [], 'prices': [], 'full_name': None, 'lat': None, 'lon': None})

for idx, row in df_products.iterrows():
    city = row['city_extracted']
    if city and city != 'Другое':
        # Находим координаты
        if city_products[city]['lat'] is None:
            lat, lon, full_name = find_city_coordinates(city)
            city_products[city]['lat'] = lat
            city_products[city]['lon'] = lon
            city_products[city]['full_name'] = full_name or city
        
        # Добавляем продукт
        city_products[city]['products'].append({
            'name': row['Unnamed: 0']
        })
        



🔍 Извлечение локаций из производителей...


In [43]:

# Удаляем города без координат
city_products = {city: data for city, data in city_products.items() 
                 if data['lat'] is not None and data['lon'] is not None}

print(f"✅ Найдено {len(city_products)} городов с координатами")


✅ Найдено 335 городов с координатами


In [38]:

# Подготовка данных для карты
map_data = []
for city, data in city_products.items():
    avg_rating = np.mean(data['ratings']) if data['ratings'] else 0
    avg_price = np.mean(data['prices']) if data['prices'] else 0
    
    map_data.append({
        'city': city,
        'full_name': data['full_name'],
        'lat': data['lat'],
        'lon': data['lon'],
        'product_count': len(data['products']),
        'avg_rating': avg_rating,
        'avg_price': avg_price,
        'products': data['products']
    })


In [39]:

# Сортировка по количеству продуктов
map_data.sort(key=lambda x: x['product_count'], reverse=True)

# Создание интерактивной карты
print("\n🗺️ Создание интерактивной карты...")



🗺️ Создание интерактивной карты...


In [48]:

# Центр карты - Москва
m = folium.Map(location=[55.7558, 37.6176], zoom_start=4, control_scale=True)

# Добавляем разные слои карты
folium.TileLayer('OpenStreetMap', name='Стандартная').add_to(m)
folium.TileLayer('CartoDB positron', name='Светлая').add_to(m)
folium.TileLayer('CartoDB dark_matter', name='Тёмная').add_to(m)

# Кластер для маркеров
marker_cluster = plugins.MarkerCluster().add_to(m)

# Добавляем маркеры для каждого города
for city_info in map_data:  # Ограничиваем 100 городов для производительности
    # Определяем размер маркера (от 1 до 40 пикселей)
    marker_size = min(1 + city_info['product_count'] // 3, 40)
    
    # Цвет маркера в зависимости от рейтинга
    if city_info['avg_rating'] >= 4.8:
        color = 'green'
        fill_color = 'darkgreen'
    elif city_info['avg_rating'] >= 4.5:
        color = 'blue'
        fill_color = 'lightblue'
    elif city_info['avg_rating'] >= 4.0:
        color = 'orange'
        fill_color = 'orange'
    else:
        color = 'red'
        fill_color = 'red'
    
    # Создаем HTML для попапа
    popup_html = f"""
    <div style="font-family: 'Segoe UI', Arial, sans-serif; min-width: 350px; max-width: 500px;">
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                    color: white; padding: 12px; border-radius: 8px 8px 0 0;">
            <h3 style="margin: 0;">🏭 {city_info['full_name']}</h3>
            <p style="margin: 5px 0 0 0; opacity: 0.9;">
                📦 {city_info['product_count']} товаров 
            </p>
        </div>
        <div style="padding: 10px; max-height: 400px; overflow-y: auto;">
    """

    
    popup_html += """
            <div style="margin-bottom: 10px;">
                <h4 style="margin: 0 0 5px 0;">📊 Категории:</h4>
                <div style="display: flex; flex-wrap: wrap; gap: 5px;">
    """
    
    
    popup_html += """
                </div>
            </div>
            <h4 style="margin: 10px 0 5px 0;">📋 Товары:</h4>
    """
    
    # Добавляем список продуктов (первые 10)
    for product in city_info['products']:
        popup_html += f"""
            <div style="border-left: 3px solid #667eea; padding: 5px 10px; margin: 5px 0; 
                        background: #f9f9f9; border-radius: 4px;">
                <strong>{product['name'][:60]}</strong><br>
            </div>
        """
    

    
    popup_html += """
        </div>
    </div>
    """
    
    # Создаем маркер
    folium.CircleMarker(
        location=[city_info['lat'], city_info['lon']],
        radius=marker_size,
        popup=folium.Popup(popup_html, max_width=500),
        tooltip=f"{city_info['full_name']}: {city_info['product_count']} товаров",
        color=color,
        fill=True,
        fillColor=fill_color,
        fillOpacity=0.7,
        weight=2
    ).add_to(marker_cluster)


In [49]:

# Добавляем тепловую карту (heatmap) для визуализации плотности
heat_data = [[city['lat'], city['lon'], city['product_count']] for city in map_data]
if heat_data:
    plugins.HeatMap(heat_data, radius=25, blur=15, max_zoom=1, min_opacity=0.5).add_to(m)

# Добавляем полный экран и поиск
plugins.Fullscreen().add_to(m)
plugins.Search(
    layer=marker_cluster,
    search_label='tooltip',
    placeholder='🔍 Поиск города...'
).add_to(m)

# Добавляем контроллер слоев
folium.LayerControl().add_to(m)

# Сохраняем карту
output_file = 'vkusvill_manufacturers_map.html'
m.save(output_file)

print(f"\n✅ Карта успешно создана: {output_file}")



✅ Карта успешно создана: vkusvill_manufacturers_map.html


In [50]:

# Статистика
print("\n📊 СТАТИСТИКА:")
print(f"   - Всего городов с производителями: {len(map_data)}")
print(f"   - Всего товаров: {sum(c['product_count'] for c in map_data)}")
total_products = sum(c['product_count'] for c in map_data)

# Топ-10 городов
print("\n🏆 ТОП-10 ГОРОДОВ ПО КОЛИЧЕСТВУ ТОВАРОВ:")
top_cities = sorted(map_data, key=lambda x: x['product_count'], reverse=True)[:10]
for i, city in enumerate(top_cities, 1):
    print(f"   {i}. {city['full_name']}: {city['product_count']} товаров (рейтинг {city['avg_rating']:.2f})")

# Сохраняем данные в CSV для анализа
summary_df = pd.DataFrame([{
    'Город': c['full_name'],
    'Количество товаров': c['product_count'],
    'Средний рейтинг': f"{c['avg_rating']:.2f}",
    'Средняя цена': f"{c['avg_price']:.0f}",
    'Широта': c['lat'],
    'Долгота': c['lon']
} for c in map_data])
summary_df.to_csv('manufacturers_by_city_detailed.csv', index=False, encoding='utf-8-sig')
print(f"\n📄 Детальная статистика сохранена: manufacturers_by_city_detailed.csv")

# Открываем карту в браузере
import webbrowser
webbrowser.open(output_file)
print(f"\n🌍 Карта открыта в браузере!")

# Вывод информации о нераспознанных городах
unmatched_cities = set()
for idx, row in df_products.iterrows():
    city = row['city_extracted']
    if city and city != 'Другое' and city not in city_products:
        unmatched_cities.add(city)

if unmatched_cities:
    print(f"\n⚠️ Не удалось найти координаты для {len(unmatched_cities)} городов:")
    for city in list(unmatched_cities)[:10]:
        print(f"   - {city}")


📊 СТАТИСТИКА:
   - Всего городов с производителями: 335
   - Всего товаров: 11059

🏆 ТОП-10 ГОРОДОВ ПО КОЛИЧЕСТВУ ТОВАРОВ:
   1. , Москва: 1974 товаров (рейтинг 0.00)
   2. Яхрома, Московская: 570 товаров (рейтинг 0.00)
   3. Саянск, Иркутская: 505 товаров (рейтинг 0.00)
   4. Черноголовка, Московская: 402 товаров (рейтинг 0.00)
   5. Химки, Московская: 323 товаров (рейтинг 0.00)
   6. Краснодар, Краснодарский: 199 товаров (рейтинг 0.00)
   7. Дмитровск, Орловская: 179 товаров (рейтинг 0.00)
   8. Подольск, Московская: 177 товаров (рейтинг 0.00)
   9. Пенза, Пензенская: 163 товаров (рейтинг 0.00)
   10. Балашиха, Московская: 153 товаров (рейтинг 0.00)

📄 Детальная статистика сохранена: manufacturers_by_city_detailed.csv

🌍 Карта открыта в браузере!

⚠️ Не удалось найти координаты для 51 городов:
   - Дрогичин
   - Абовян
   - Индия
   - Венгрия
   - Туркестанская
   - Новогрудок
   - Муниципальный
   - Россия
   - Таиланд
   - Тебриз


In [53]:
import pandas as pd
import re
from collections import defaultdict

# Загрузка данных
df = pd.read_csv('vkusvill_itog_2026-05-18.csv', sep=',')

def extract_multiple_manufacturers(manufacturer_str):
    """
    Извлекает список производителей из строки, где может быть несколько адресов.
    Пример: "ООО "ЕЛЕЦКИЙ МЯСОКОМБИНАТ":Россия, 399780... ОАО "ВНМД":Россия, 173008..."
    """
    if pd.isna(manufacturer_str):
        return []
    
    manufacturers = []
    
    # Паттерн для поиска организаций: название организации, затем двоеточие и адрес
    # Ищем: "НАЗВАНИЕ ОРГАНИЗАЦИИ": адрес
    pattern = r'([А-ЯЁ][А-ЯЁ\s\-"]+?[А-ЯЁ]):\s*([^;:]+?(?:область|край|Республика|г\.|город)[^;:]+?)(?=\s*[А-ЯЁ][А-ЯЁ\s\-"]+?:|$)'
    
    matches = re.findall(pattern, manufacturer_str, re.DOTALL)
    
    for match in matches:
        org_name = match[0].strip()
        address = match[1].strip()
        # Очищаем адрес от лишних символов
        address = re.sub(r'[;"]', '', address).strip()
        manufacturers.append({
            'organization': org_name,
            'address': address
        })
    
    # Если не нашли по сложному паттерну, пробуем простой - по разделителю ;
    if not manufacturers and ';' in manufacturer_str:
        parts = manufacturer_str.split(';')
        for part in parts:
            if ':' in part:
                org, addr = part.split(':', 1)
                manufacturers.append({
                    'organization': org.strip(),
                    'address': addr.strip()
                })
    
    return manufacturers

# Применяем функцию к датасету
print("🔍 Парсинг производителей...")
df['manufacturers_parsed'] = df['manufacturer'].apply(extract_multiple_manufacturers)
df['manufacturers_count'] = df['manufacturers_parsed'].apply(len)

# Статистика по количеству производителей
print("\n📊 СТАТИСТИКА:")
print(f"   - Всего товаров: {len(df)}")
print(f"   - Товаров с 1 производителем: {(df['manufacturers_count'] == 1).sum()}")
print(f"   - Товаров с 2+ производителями: {(df['manufacturers_count'] >= 2).sum()}")
print(f"   - Максимальное число производителей у одного товара: {df['manufacturers_count'].max()}")

# Топ-10 продуктов с самым большим количеством производителей
print("\n🏆 ТОП-10 ПРОДУКТОВ С НАИБОЛЬШИМ КОЛИЧЕСТВОМ ПРОИЗВОДИТЕЛЕЙ:")
print("=" * 80)

top_products = df.nlargest(10, 'manufacturers_count')[['Unnamed: 0', 'manufacturers_count', 'manufacturers_parsed', 'category']]

for idx, (i, row) in enumerate(top_products.iterrows(), 1):
    print(f"\n{idx}. 📦 {row['Unnamed: 0']}")
    print(f"   Категория: {row['category']}")
    print(f"   Количество производителей: {row['manufacturers_count']}")
    print("   Производители:")
    for j, m in enumerate(row['manufacturers_parsed'], 1):
        print(f"      {j}. {m['organization']}")
        print(f"         Адрес: {m['address'][:80]}...")

# Детальная таблица для экспорта
print("\n📄 Экспорт данных...")
top_products_detail = []
for idx, row in top_products.iterrows():
    for m in row['manufacturers_parsed']:
        top_products_detail.append({
            'Товар': row['Unnamed: 0'],
            'Категория': row['category'],
            'Всего производителей': row['manufacturers_count'],
            'Производитель': m['organization'],
            'Адрес': m['address']
        })

detail_df = pd.DataFrame(top_products_detail)
detail_df.to_csv('products_with_multiple_manufacturers.csv', index=False, encoding='utf-8-sig')
print("✅ Сохранено в products_with_multiple_manufacturers.csv")

# Анализ: какие производители чаще всего встречаются вместе
print("\n🔗 АНАЛИЗ ПАР ПРОИЗВОДИТЕЛЕЙ, ВСТРЕЧАЮЩИХСЯ ВМЕСТЕ:")

from itertools import combinations

pair_counts = defaultdict(int)
for manufacturers in df['manufacturers_parsed']:
    if len(manufacturers) >= 2:
        org_names = [m['organization'] for m in manufacturers]
        for pair in combinations(sorted(org_names), 2):
            pair_counts[pair] += 1

top_pairs = sorted(pair_counts.items(), key=lambda x: x[1], reverse=True)[:10]
for pair, count in top_pairs:
    print(f"   {pair[0]} + {pair[1]}: {count} товаров")

🔍 Парсинг производителей...

📊 СТАТИСТИКА:
   - Всего товаров: 20165
   - Товаров с 1 производителем: 996
   - Товаров с 2+ производителями: 49
   - Максимальное число производителей у одного товара: 3

🏆 ТОП-10 ПРОДУКТОВ С НАИБОЛЬШИМ КОЛИЧЕСТВОМ ПРОИЗВОДИТЕЛЕЙ:

1. 📦 Фарш из голени индейки
   Категория: Мясо, птица
   Количество производителей: 3
   Производители:
      1. ООО
         Адрес: Россия, 442151, Пензенская область, Нижнеломовский район, село Овчарное, ул. Луг...
      2. ООО
         Адрес: Россия, 346473, Ростовская область, м.р-н Октябрьский, п. Интернациональный, с.п...
      3. ООО
         Адрес: Россия, 193149, Ленинградская область, Всеволожский муниципальный район, Свердло...

2. 📦 Пюре из груши, банана и киви, 90 г
   Категория: Товары для детей
   Количество производителей: 2
   Производители:
      1. ОАО
         Адрес: Республика Беларусь, 225903, Брестская обл., г. Малорита, ул. Заводская, 9....
      2. ФРУКТИС ООО
         Адрес: 353793, Краснодарский край